In [ ]:
import numpy as np
import pandas as pd

# Constants and parameters
WINDTURBINE_PRODUCED_YEARLY = 24_054_796.56
HOME_COUNT = 5000
MEAN_PERSONS = 4
STD_PERSONS = 1
MIN_PERSONS = 1
MAX_PERSONS = 6

MONTHLY_USAGE_KWH = {
    ("apartment", 1): 121,
    ("house", 1): 183,
    ("apartment", 2): 188,
    ("house", 2): 258,
    ("apartment", 3): 246,
    ("house", 3): 325,
    ("apartment", 4): 292,
    ("house", 4): 383,
    ("apartment", 5): 342,
    ("house", 5): 442,
    ("apartment", 6): 383,
    ("house", 6): 500,
}

month_names = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec",
]

# Generate synthetic data for 5000 homes
rng = np.random.default_rng(42)
home_types = rng.choice(["house", "apartment"], size=HOME_COUNT)
persons = np.clip(
    np.rint(rng.normal(loc=MEAN_PERSONS, scale=STD_PERSONS, size=HOME_COUNT)).astype(int),
    MIN_PERSONS,
    MAX_PERSONS,
)

# Create a list of dictionaries to hold the data for each home
rows = []
for index, (home_type, person_count) in enumerate(zip(home_types, persons), start=1):
    avg_monthly_kwh = MONTHLY_USAGE_KWH[(home_type, int(person_count))]

    monthly_adjustments = []
    for month in range(1, 13):
        if month in (3, 4):
            adjustment = -0.15
        elif month in (12, 1, 2):
            adjustment = rng.uniform(0.00, 0.07)
        elif month in (4, 5, 9, 10):
            adjustment = rng.uniform(-0.15, 0.0)
        elif month in (6, 7, 8):
            adjustment = rng.uniform(-0.15, -0.05)    
        else:
            adjustment = rng.uniform(-0.03, 0.03)
        monthly_adjustments.append(adjustment)

    monthly_usage = [avg_monthly_kwh * (1 + adjustment) for adjustment in monthly_adjustments]
    yearly_kwh = float(np.sum(monthly_usage))

    row = {
        "home_id": f"home_{index}",
        "home_type": home_type,
        "persons": int(person_count),
        "average_monthly_usage_kWh": avg_monthly_kwh,
        "yearly_usage_kWh": yearly_kwh,
    }

    for month_name, usage_value in zip(month_names, monthly_usage):
        row[f"{month_name}_usage_kWh"] = usage_value

    rows.append(row)

# Create a DataFrame from the list of dictionaries
homes_df = pd.DataFrame(rows)

# Calculate total energy usage for each month across all homes
monthly_columns = [f"{month_name}_usage_kWh" for month_name in month_names]
combined_monthly_usage_kwh = homes_df[monthly_columns].sum()
homes_shared_energy_usage = combined_monthly_usage_kwh.copy()

for month_name in month_names:
    print(f"{month_name}: {homes_shared_energy_usage[f'{month_name}_usage_kWh']:,.2f} kWh")

jan: 1,752,164.21 kWh
feb: 1,751,765.32 kWh
mar: 1,438,480.50 kWh
apr: 1,438,480.50 kWh
may: 1,565,116.40 kWh
jun: 1,522,588.21 kWh
jul: 1,523,151.08 kWh
aug: 1,523,324.26 kWh
sep: 1,564,995.01 kWh
oct: 1,565,316.25 kWh
nov: 1,692,342.98 kWh
dec: 1,752,415.58 kWh


In [9]:
# Total yearly usage for all homes in the current homes_df
total_all_homes_yearly_usage_kwh = homes_df["yearly_usage_kWh"].sum()

print(f"Total yearly usage for all homes: {total_all_homes_yearly_usage_kwh:,.2f} kWh")
total_all_homes_yearly_usage_kwh

Total yearly usage for all homes: 19,090,140.31 kWh


np.float64(19090140.30661267)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display

month_day_counts = {
    "jan": 31,
    "feb": 28,
    "mar": 31,
    "apr": 30,
    "may": 31,
    "jun": 30,
    "jul": 31,
    "aug": 31,
    "sep": 30,
    "oct": 31,
    "nov": 30,
    "dec": 31,
}

# Determine the source of monthly totals
if "homes_shared_energy_usage" in globals():
    monthly_totals = homes_shared_energy_usage
elif "combined_monthly_usage_kwh" in globals():
    monthly_totals = combined_monthly_usage_kwh
elif "homes_df" in globals():
    month_names = list(month_day_counts.keys())
    monthly_columns = [f"{month_name}_usage_kWh" for month_name in month_names]
    monthly_totals = homes_df[monthly_columns].sum()
else:
    monthly_totals = pd.Series({f"{month}_usage_kWh": 0.0 for month in month_day_counts})
    print("Warning: No monthly source found. Using 0.0 kWh for all months.")

# Generate a set of daily usage from monthly totals
rng = np.random.default_rng(42)

daily_usage_by_month = {}
rows = []

for month, day_count in month_day_counts.items():
    monthly_key = f"{month}_usage_kWh"
    month_total = float(monthly_totals[monthly_key])

    random_weights = rng.random(day_count)
    daily_values = (random_weights / random_weights.sum()) * month_total

    daily_values = np.round(daily_values, 2)
    correction = np.round(month_total - daily_values.sum(), 2)
    daily_values[-1] = np.round(daily_values[-1] + correction, 2)

    month_df = pd.DataFrame({
        "month": month,
        "day": np.arange(1, day_count + 1),
        "usage_kWh": daily_values,
    })

    daily_usage_by_month[month] = month_df
    rows.append(month_df)

all_daily_usage = pd.concat(rows, ignore_index=True)

# Build a daily timeline for 2025 and let month update automatically in the loop
simulation_daily_demand = all_daily_usage.copy()
simulation_daily_demand["date"] = pd.date_range("2025-01-01", "2025-12-31", freq="D")

# Example simulation loop (replace placeholder logic with battery/wind logic later)
daily_demand_rows = []

for row in simulation_daily_demand.itertuples(index=False):
    current_date = row.date
    homes_shared_energy_usage_kwh_day = float(row.usage_kWh)

    daily_demand_rows.append({
        "date": current_date.strftime("%Y-%m-%d %H:%M:%S"),
        "homes_shared_energy_usage_kWh": homes_shared_energy_usage_kwh_day,
    })

# Create a DataFrame from the daily demand rows
simulation_df = pd.DataFrame(daily_demand_rows)

# Create visualization with scrollable HTML table
scrollable_html = (
    '<div style="max-height: 500px; overflow-y: auto; border: 1px solid #ddd;">'
    + simulation_df.to_html(index=False)
    + "</div>"
 )
display(HTML(scrollable_html))

date,homes_shared_energy_usage_kWh
2025-01-01 00:00:00,77407.61
2025-01-02 00:00:00,43894.65
2025-01-03 00:00:00,85873.11
2025-01-04 00:00:00,69747.62
2025-01-05 00:00:00,9419.20
2025-01-06 00:00:00,97577.37
2025-01-07 00:00:00,76125.78
2025-01-08 00:00:00,78618.62
2025-01-09 00:00:00,12813.35
2025-01-10 00:00:00,45045.58


In [ ]:
# Save the daily demand DataFrame to an Excel file
simulation_df.to_excel("Daily_Usage_Homes.xlsx", index=False)